# Feature Engineering

Prototype the offline feature-engineering pipeline for the Fashion Recommendation System.
This notebook stages sampled Parquet data to S3-compatible storage and builds
**point-in-time** anchor-pair features on the **entire** dataset (no train/val/test split).

Each row is featurized at a **snap date** using purchase history strictly on or before that cutoff.

| Stage | Local path | AWS path |
|-------|------------|----------|
| Source datasets | `dataset/dummy`, `dataset/sample_2000_users` | same (uploaded) |
| Staged raw | `s3/dataset/{name}/` | `s3://{bucket}/dataset/{name}/` |
| Features | `s3/dataset/{name}/features/` (Hive `snap_date=...`) | `s3://{bucket}/dataset/{name}/features/` |

Feature definitions: [`features-eng.md`](../docs/implementation-info/guides/features-eng.md)  
Schema: [`schema-info.md`](../docs/system-design/schema-info.md)  
Requirements: [`v1-requirements.md`](../docs/system-design/v1/v1-requirements.md)

Runs on **local PySpark** (`local[*]`). Configuration lives in `configs/**/*.yaml`;
infrastructure env vars live in `src/fashion_recommendation_system/config.py` (used by
`pipelines/run_feature_pipeline.py`). This notebook loads YAML via `notebooks/utils/config_loader.py`
(does not import from `src/` per project-structure.md).

Spark session setup: `notebooks/utils/spark_session.py`. Core builders:
`notebooks/utils/feature_engineering_core.py` (imported below).

**Kernel:** `Fashion Reco (notebooks)`


## 1. Configuration

Loads merged settings from `configs/data/` and `configs/features/`. To change the active
dataset, edit `configs/data/s3_paths.yaml` → `datasets.active` (e.g. `dummy` for smoke tests).
For AWS/Glue production runs, use `pipelines/run_feature_pipeline.py` (reads `config.py` for infra).

In [1]:
%load_ext autoreload
%autoreload 2

import json
import math
import os
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

for _nb in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (_nb / "utils").is_dir():
        sys.path[:0] = [str(_nb)]
        break

from utils.config_loader import is_glue_runtime, load_feature_engineering_config

CONFIG = load_feature_engineering_config(environment="local_dev")

STORAGE_MODE = CONFIG["storage_mode"]       # "local" → file:///s3/ mirror, "aws" → real S3
STAGE_DATASETS = CONFIG["stage_datasets"]   # e.g. ["dummy", "sample_2000_users"]
IS_GLUE = is_glue_runtime(CONFIG["runtime_mode"])  # True when running on AWS Glue
CONFIG

{'repo_root': 'F:\\git-projects\\fashion-recommendation-system',
 'environment': 'local_dev',
 'storage_mode': 'local',
 'dataset_name': 'sample_2000_users',
 'stage_datasets': ['dummy', 'sample_2000_users'],
 'local_dataset_root': 'dataset',
 'local_s3_root': 's3',
 's3_bucket': 'fashion-reco-dev',
 'aws_region': 'us-east-1',
 'localstack_endpoint': '',
 'train_end': '2020-03-24',
 'cutoff_date': '2020-03-31',
 'test_end': '2020-04-07',
 'feature_cutoff': '2020-03-24',
 'label_window_days': 7,
 'category_col': 'garment_group_name',
 'color_col': 'colour_group_name',
 'decay_half_life_days': 180,
 'item_lookbacks_days': {'short': 7, 'medium': 30, 'long': 180},
 'user_pref_top_n': 3,
 'user_pref_lookback_days': 365,
 'feature_sets': {'item': True,
  'user': True,
  'cross': True,
  'transaction_globals': True},
 'prefixes': {'staged': 'dataset/{dataset_name}',
  'splits': 'dataset/{dataset_name}/splits',
  'features': 'dataset/{dataset_name}/features'},
 'runtime_mode': 'local',
 'cross

## 2. Spark Session

Local PySpark driver (or reuse Glue-provided `spark`) via `notebooks/utils/spark_session.py`.
Includes Windows Hadoop shim for Parquet writes — see [`java-pyspark-local-setup.md`](../docs/implementation-info/guides/java-pyspark-local-setup.md).

In [2]:
from utils.spark_session import create_spark_session

spark = create_spark_session("feature-engineering", is_glue=IS_GLUE, pin_python=True)

## 3. Storage Helpers

Abstract local `s3/` mirror vs real S3 so downstream Spark reads use the same relative keys.

In [3]:
def resolve_repo_root() -> Path:
    """Find repository root by locating requirements-notebooks.txt."""
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "requirements-notebooks.txt").exists():
            return candidate
    return Path.cwd()

def storage_uri(relative_path: str) -> str:
    """Map a relative data-lake key to a local or S3 URI."""
    relative_path = relative_path.strip("/").replace("\\", "/")
    if STORAGE_MODE == "aws":
        return f"s3://{CONFIG['s3_bucket']}/{relative_path}"
    local_root = (resolve_repo_root() / CONFIG["local_s3_root"]).resolve()
    return str((local_root / relative_path).resolve())

def local_source_dataset_path(dataset_name: str) -> Path:
    """Resolve on-disk Parquet source under dataset/."""
    return (resolve_repo_root() / CONFIG["local_dataset_root"] / dataset_name).resolve()

def _replace_local_path(src_item: Path, dest_item: Path) -> None:
    """Copy one file or directory tree, replacing *dest_item* if it already exists."""
    if dest_item.exists():
        if dest_item.is_dir():
            shutil.rmtree(dest_item)
        else:
            dest_item.unlink()
    if src_item.is_dir():
        shutil.copytree(src_item, dest_item)
    else:
        dest_item.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_item, dest_item)


def copy_dataset_tables_local(src: Path, dest: Path) -> None:
    """Stage raw tables from dataset/ without deleting downstream artifacts.

    Only paths present under *src* (e.g. articles, customers, transactions) are
    replaced in *dest*. Sibling paths already staged elsewhere — such as
    ``transactions_with_label`` from notebook 02 or ``features/`` from this
    notebook — are left untouched.

    Each replaced path is removed first so stale PySpark part-files never
    accumulate alongside a new single-file Parquet (mixed files cause Spark
    schema-merge failures).
    """
    if not src.exists():
        raise FileNotFoundError(f"Source dataset not found: {src}")
    dest.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        _replace_local_path(item, dest / item.name)

def upload_tree_to_s3(local_src: Path, s3_prefix: str) -> int:
    """Upload a local directory tree to S3 using boto3."""
    import boto3
    if not local_src.exists():
        raise FileNotFoundError(f"Source dataset not found: {local_src}")
    client_kwargs = {"region_name": CONFIG["aws_region"]}
    if CONFIG["localstack_endpoint"]:
        client_kwargs["endpoint_url"] = CONFIG["localstack_endpoint"]
    s3 = boto3.client("s3", **client_kwargs)
    uploaded = 0
    for path in local_src.rglob("*"):
        if path.is_file():
            key = f"{s3_prefix.strip('/')}/{path.relative_to(local_src).as_posix()}"
            s3.upload_file(str(path), CONFIG["s3_bucket"], key)
            uploaded += 1
    return uploaded

def stage_dataset(dataset_name: str) -> dict:
    """Stage one dataset from dataset/ to S3-compatible storage."""
    src = local_source_dataset_path(dataset_name)
    relative_dest = f"dataset/{dataset_name}"
    if STORAGE_MODE == "aws":
        files = upload_tree_to_s3(src, relative_dest)
        destination = storage_uri(relative_dest)
    else:
        dest = Path(storage_uri(relative_dest))
        copy_dataset_tables_local(src, dest)
        files = sum(1 for p in src.rglob("*") if p.is_file())
        destination = str(dest)
    return {
        "dataset": dataset_name,
        "mode": STORAGE_MODE,
        "source": str(src),
        "destination": destination,
        "files": files,
    }



### 3.1 Stage datasets to S3

Copy raw tables from `dataset/dummy` and `dataset/sample_2000_users` into the
S3-compatible layout. Re-running replaces only source tables (`articles`,
`customers`, `transactions`) and **preserves** downstream paths such as
`transactions_with_label/` (notebook 02) and `features/` (this notebook).

In [4]:
# Stage every dataset listed in config (typically dummy + sample_2000_users).
stage_results = [stage_dataset(name) for name in STAGE_DATASETS]
stage_results

[{'dataset': 'dummy',
  'mode': 'local',
  'source': 'F:\\git-projects\\fashion-recommendation-system\\dataset\\dummy',
  'destination': 'F:\\git-projects\\fashion-recommendation-system\\s3\\dataset\\dummy',
  'files': 11},
 {'dataset': 'sample_2000_users',
  'mode': 'local',
  'source': 'F:\\git-projects\\fashion-recommendation-system\\dataset\\sample_2000_users',
  'destination': 'F:\\git-projects\\fashion-recommendation-system\\s3\\dataset\\sample_2000_users',
  'files': 43}]

## 4. Load Staged Data

Read Parquet tables from the staged location for the active dataset (`CONFIG['dataset_name']`).

In [5]:
def read_staged_table(spark: SparkSession, dataset_name: str, table: str) -> DataFrame:
    """Read a staged Parquet table (articles, customers, or transactions).

    Parameters
    ----------
    spark : SparkSession
        Active Spark session.
    dataset_name : str
        Dataset folder name under ``dataset/``.
    table : str
        Table subfolder: ``articles``, ``customers``, or ``transactions``.

    Returns
    -------
    DataFrame
        Parquet DataFrame with schema from the sampling notebook.
    """
    path = storage_uri(f"dataset/{dataset_name}/{table}")
    # Spark resolves file:// or s3:// via storage_uri — same code path for local and AWS.
    return spark.read.parquet(path)


# Active dataset comes from configs/data/s3_paths.yaml → datasets.active
DATASET = CONFIG["dataset_name"]

# Read the three raw Parquet tables from the staged data-lake path.
articles_df = read_staged_table(spark, DATASET, "articles")
customers_df = read_staged_table(spark, DATASET, "customers")
transactions_df = read_staged_table(spark, DATASET, "transactions")

# Row counts — sanity check before feature work.
print(f"articles: {articles_df.count():,}")
print(f"customers: {customers_df.count():,}")
print(f"transactions: {transactions_df.count():,}")

articles: 22,186
customers: 2,000


transactions: 48,265


### 4.1 Parquet write helpers

Shared helpers for persisting enriched transaction features.

In [6]:
def write_parquet_dataset(df: DataFrame, relative_path: str, partition_cols: list = None) -> str:
    """Write DataFrame to Parquet, handling local vs S3 paths.

    Parameters
    ----------
    df : DataFrame
        Data to write.
    relative_path : str
        Path relative to data lake root, e.g. ``dataset/sample/features`` (Hive ``snap_date=...``).
    partition_cols : list, optional
        Columns to partition by, by default None.

    Returns
    -------
    str
        The resolved URI where data was written.
    """
    uri = storage_uri(relative_path)
    writer = df.write.mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(uri)
    return uri


## 5. Point-in-Time Feature Engineering

1. Load the ranking dataset (`transactions_with_label`) containing positives and window-aware negatives.
2. Compute features for these `(customer_id, article_id)` pairs using history strictly `<= snap_date`.

All features are defined in [`features-eng.md`](../docs/implementation-info/guides/features-eng.md).

### Feature families

| Family | Key features |
|--------|-------------|
| **Item popularity** | `item_pop_7d/30d/180d`, category pops, `item_pop_same_month_last_year` |
| **Item demand ratios** | `item_recent_to_last_180d_ratio`, `item_category_recent_to_lifetime_ratio`, `item_seasonality_strength` |
| **Item catalog** | `first_sold_date`, `days_since_first_sold` |
| **Item price/recency/channel** *(new)* | `item_avg_price`, `item_days_since_last_sold`, `item_sales_channel_2_count`, `item_sales_channel_2_share` |
| **User preferences** | `user_category_pref_1y_rank1/2/3`, `user_color_pref_1y_rank1/2` |
| **User activity** | `user_days_since_last_purchase`, `user_purchase_count_30d/180d` |
| **User price** | `user_decayed_price_avg`, `user_decayed_price_std` |
| **User–item lifetime** | `user_item_repurchase`, `user_item_decayed_repurchase`, `user_item_decayed_interaction_ratio`, `user_item_price_decayed_zscore` |
| **User–item windowed** *(new)* | `user_item_repurchase_30d/90d/365d`, `user_item_days_since_last_purchase`, `user_item_sales_channel_2_count` |
| **User–category** *(new)* | `user_purchases_in_candidate_category_1y`, `user_days_since_last_purchase_in_category`, `user_category_match_rank1/2/3` |
| **Time context** | `txn_month_sin`, `txn_month_cos` |


In [7]:
from utils.feature_engineering_core import build_features

# Load anchors from the staged transactions_with_label dataset
anchors_df = read_staged_table(spark, DATASET, "transactions_with_label")

# Build features
enriched_txn_df = build_features(
    anchors_df,
    transactions_df,
    articles_df,
    customers_df,
    CONFIG["category_col"],
    CONFIG["color_col"],
    CONFIG["decay_half_life_days"],
    color_pref_top_n=2,
    category_pref_top_n=CONFIG["user_pref_top_n"],
)

print(f"anchors (pos + neg): {anchors_df.count():,}")
print(f"enriched features: {enriched_txn_df.count():,}")
print(f"total columns: {len(enriched_txn_df.columns)}")

# ── Existing features ────────────────────────────────────────────────
print("\n── Item popularity + demand ratios ──")
enriched_txn_df.select(
    "customer_id", "article_id", "snap_date", "label",
    "item_pop_7d", "item_pop_30d", "item_pop_180d",
    "item_recent_to_last_180d_ratio",
    "item_seasonality_strength",
).show(5, truncate=False)

# ── New item features ────────────────────────────────────────────────
print("\n── New: item price / recency / channel ──")
enriched_txn_df.select(
    "customer_id", "article_id", "snap_date",
    "item_avg_price",
    "item_days_since_last_sold",
    "item_sales_channel_2_count",
    "item_sales_channel_2_share",
).show(5, truncate=False)

# ── New user–item windowed cross features ────────────────────────────
print("\n── New: user–item windowed repurchase + pair recency ──")
enriched_txn_df.select(
    "customer_id", "article_id", "snap_date",
    "user_item_repurchase",
    "user_item_repurchase_30d",
    "user_item_repurchase_90d",
    "user_item_repurchase_365d",
    "user_item_days_since_last_purchase",
    "user_item_sales_channel_2_count",
).show(5, truncate=False)

# ── New user–category cross features ─────────────────────────────────
print("\n── New: user–category cross features ──")
enriched_txn_df.select(
    "customer_id", "article_id", "snap_date",
    "item_category",
    "user_category_pref_1y_rank1",
    "user_purchases_in_candidate_category_1y",
    "user_days_since_last_purchase_in_category",
    "user_category_match_rank1",
    "user_category_match_rank2",
    "user_category_match_rank3",
).show(5, truncate=False)


anchors (pos + neg): 51,557


enriched features: 51,557
total columns: 76

── Item popularity + demand ratios ──


+----------------------------------------------------------------+----------+----------+-----+-----------+------------+-------------+------------------------------+-------------------------+
|customer_id                                                     |article_id|snap_date |label|item_pop_7d|item_pop_30d|item_pop_180d|item_recent_to_last_180d_ratio|item_seasonality_strength|
+----------------------------------------------------------------+----------+----------+-----+-----------+------------+-------------+------------------------------+-------------------------+
|062558d013f6c1e93edb61a9e5de02a2526ba39b0f5a3fb1904c2de0282e57d0|0889503001|2020-06-30|1    |0          |0           |0            |0.0                           |0.0                      |
|111af600e164ebeb48231848949144dfac1990a3f814e831990f952ee61e150a|0810170012|2020-06-30|1    |0          |0           |0            |0.0                           |0.0                      |
|142a7205992a36d93a8406b31fb550489fcbe160ea3e

+----------------------------------------------------------------+----------+----------+--------------------+-------------------------+--------------------------+--------------------------+
|customer_id                                                     |article_id|snap_date |item_avg_price      |item_days_since_last_sold|item_sales_channel_2_count|item_sales_channel_2_share|
+----------------------------------------------------------------+----------+----------+--------------------+-------------------------+--------------------------+--------------------------+
|062558d013f6c1e93edb61a9e5de02a2526ba39b0f5a3fb1904c2de0282e57d0|0889503001|2020-06-30|null                |null                     |0                         |0.0                       |
|142a7205992a36d93a8406b31fb550489fcbe160ea3ead3a78a5e74b3ee7916c|0838769002|2020-06-30|0.024135593324899673|46                       |2                         |0.6666666666666666        |
|142a7205992a36d93a8406b31fb550489fcbe160ea3ead3a7

+----------------------------------------------------------------+----------+----------+--------------------+------------------------+------------------------+-------------------------+----------------------------------+-------------------------------+
|customer_id                                                     |article_id|snap_date |user_item_repurchase|user_item_repurchase_30d|user_item_repurchase_90d|user_item_repurchase_365d|user_item_days_since_last_purchase|user_item_sales_channel_2_count|
+----------------------------------------------------------------+----------+----------+--------------------+------------------------+------------------------+-------------------------+----------------------------------+-------------------------------+
|062558d013f6c1e93edb61a9e5de02a2526ba39b0f5a3fb1904c2de0282e57d0|0889503001|2020-06-30|0                   |0                       |0                       |0                        |null                              |0                    

+----------------------------------------------------------------+----------+----------+--------------+---------------------------+---------------------------------------+-----------------------------------------+-------------------------+-------------------------+-------------------------+
|customer_id                                                     |article_id|snap_date |item_category |user_category_pref_1y_rank1|user_purchases_in_candidate_category_1y|user_days_since_last_purchase_in_category|user_category_match_rank1|user_category_match_rank2|user_category_match_rank3|
+----------------------------------------------------------------+----------+----------+--------------+---------------------------+---------------------------------------+-----------------------------------------+-------------------------+-------------------------+-------------------------+
|062558d013f6c1e93edb61a9e5de02a2526ba39b0f5a3fb1904c2de0282e57d0|0889503001|2020-06-30|Accessories   |Dresses Ladies       

## 6. Post-FE Imputation and Sparsity Filters

As per `pre-processing-guide.md`:
- **Post-FE Imputation:** Fill count features and cross-feature nulls with 0 default imputation. Fill item average price null with 0.
- **Sparsity & Active-Item Filters:** Drop ranking pair rows where `item_pop_30d` equals zero (dead SKUs filter).

In [8]:
count_cols = [
    "item_pop_7d", "item_pop_30d", "item_pop_180d",
    "item_category_pop_30d", "item_category_pop_180d",
    "item_pop_same_month_last_year",
    "item_sales_channel_2_count",
    "user_purchase_count_30d", "user_purchase_count_180d",
    "user_item_repurchase", "user_item_repurchase_30d",
    "user_item_repurchase_90d", "user_item_repurchase_365d",
    "user_item_sales_channel_2_count",
    "user_purchases_in_candidate_category_1y",
    "user_item_decayed_repurchase",
    "item_avg_price"
]

# Fill nulls with 0
enriched_txn_df = enriched_txn_df.fillna(0, subset=count_cols)

# Apply sparsity filter: drop dead SKUs
enriched_txn_df = enriched_txn_df.filter(F.col("item_pop_30d") > 0)

print(f"enriched features after filters: {enriched_txn_df.count():,}")


enriched features after filters: 4,278


## 7. Persist Feature Outputs

Write the enriched transaction table under `dataset/{name}/features/`.

In [9]:
features_base = f"dataset/{DATASET}/features"
# Partition by snap_date for easy loading
feature_outputs = {
    "features": write_parquet_dataset(
        enriched_txn_df,
        features_base,
        partition_cols=["snap_date"],
    ),
}
feature_outputs


{'features': 'F:\\git-projects\\fashion-recommendation-system\\s3\\dataset\\sample_2000_users\\features'}

## 8. Run Summary


In [10]:
summary = {
    "run_at_utc": datetime.now(timezone.utc).isoformat(),
    "storage_mode": CONFIG["storage_mode"],
    "dataset": DATASET,
    "transaction_rows": enriched_txn_df.count(),
    "feature_columns": enriched_txn_df.columns,
    "staged_datasets": stage_results,
    "feature_outputs": feature_outputs,
}
print(json.dumps(summary, indent=2))


{
  "run_at_utc": "2026-07-03T09:02:10.209559+00:00",
  "storage_mode": "local",
  "dataset": "sample_2000_users",
  "transaction_rows": 4278,
  "feature_columns": [
    "customer_id",
    "article_id",
    "snap_date",
    "label",
    "FN",
    "Active",
    "club_member_status",
    "fashion_news_frequency",
    "age",
    "postal_code",
    "product_code",
    "prod_name",
    "product_type_no",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_no",
    "graphical_appearance_name",
    "colour_group_code",
    "perceived_colour_value_id",
    "perceived_colour_value_name",
    "perceived_colour_master_id",
    "perceived_colour_master_name",
    "department_no",
    "department_name",
    "index_code",
    "index_name",
    "index_group_no",
    "index_group_name",
    "section_no",
    "section_name",
    "garment_group_no",
    "detail_desc",
    "item_category",
    "item_color",
    "item_pop_7d",
    "item_pop_30d",
    "item_pop_180d",
    "item_cat